# Publication notes

This is the original exploratory workflow with saved outputs removed and paths made configurable. Collection and analytical logic is retained. Source websites and identifiers may change; notebooks 01–02 are an inspected research workflow, not a guaranteed unattended scraper. Review collection permissions and source-specific settings before recollection. Do not run against your only copy of historical data: original export cells can overwrite files.

Run from a clean kernel in order. Data is local and excluded from GitHub. Historical saved review text and reviewer details are not published as notebook outputs.


In [ ]:
from pathlib import Path
import os

# Configure KBEAUTY_PROJECT_ROOT to keep data outside the cloned repository.
_working_dir = Path.cwd().resolve()
_default_root = _working_dir.parent if _working_dir.name == "notebooks" else _working_dir
PROJECT_ROOT = Path(os.environ.get("KBEAUTY_PROJECT_ROOT", str(_default_root))).expanduser().resolve()
# Original collection cells use relative data/ paths.
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


In [ ]:
from pathlib import Path
import pandas as pd

data_folder = (
    PROJECT_ROOT
    / "data"
    / "cleaned"
    / "all_products"
)

products = pd.read_csv(
    data_folder / "products.csv",
    dtype={"product_id": str, "sku": str}
)

reviews = pd.read_csv(
    data_folder / "reviews.csv",
    dtype={
        "product_id": str,
        "review_id": str,
        "incentivized_standardized": "boolean"
    }
)

reviews["rating"] = pd.to_numeric(reviews["rating"], errors="raise")
assert reviews["rating"].between(1, 5).all()

analysis_reviews = reviews.merge(
    products[["product_id", "brand", "product_name", "analysis_segment"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

assert analysis_reviews["brand"].notna().all()

# Overall ratings and incentive coverage for each product.
product_summary = (
    analysis_reviews.groupby(
        ["product_id", "brand", "product_name", "analysis_segment"]
    )
    .agg(
        review_count=("review_id", "size"),
        average_rating=("rating", "mean"),
        median_rating=("rating", "median"),
        low_rating_pct=("rating", lambda x: x.le(2).mean() * 100),
        five_star_pct=("rating", lambda x: x.eq(5).mean() * 100),
        incentive_known_count=("incentivized_standardized", "count"),
        incentivized_count=("incentivized_standardized", "sum")
    )
    .reset_index()
)

product_summary["incentivized_pct"] = (
    100 * product_summary["incentivized_count"]
    / product_summary["incentive_known_count"]
)

# Compare ratings within each product by incentive status.
analysis_reviews["incentive_group"] = (
    analysis_reviews["incentivized_standardized"]
    .map({True: "Incentivized", False: "Not incentivized"})
    .fillna("Unknown")
)

incentive_summary = (
    analysis_reviews.groupby(
        ["brand", "product_name", "incentive_group"]
    )
    .agg(
        review_count=("review_id", "size"),
        average_rating=("rating", "mean"),
        low_rating_pct=("rating", lambda x: x.le(2).mean() * 100)
    )
    .reset_index()
)

print("PRODUCT RATINGS")
print(product_summary.round(2).to_string(index=False))

print("\nRATINGS BY INCENTIVE STATUS")
print(incentive_summary.round(2).to_string(index=False))

In [ ]:
# Inspect the reviews behind the COSRX serum rating gap.
mask = (
    analysis_reviews["product_id"].eq("COSRX_SERUM_01")
    & analysis_reviews["incentivized_standardized"].eq(False).fillna(False)
    & analysis_reviews["rating"].le(2)
)

cosrx_serum_low_reviews = (
    analysis_reviews.loc[mask]
    .sort_values("review_date", ascending=False)
    .copy()
)

print("Reviews to inspect:", len(cosrx_serum_low_reviews))

for _, review in cosrx_serum_low_reviews.iterrows():
    print("\n" + "-" * 70)
    print("Review ID:", review["review_id"])
    print("Date:", review["review_date"])
    print("Rating:", review["rating"])
    print("Language:", review.get("language"))
    print("Verified:", review.get("verified_standardized"))
    print("Skin type:", review.get("skin_type_standardized"))
    print("Title:", review.get("review_title"))
    print("Text:", review.get("review_text"))

In [ ]:
# Manual coding based on the eight full review texts.
# Themes describe reported experiences, not established causation.
review_codes = {
    "yotpo_839945794": ["Reported breakouts"],
    "yotpo_819855828": ["Reported breakouts"],
    "yotpo_811656900": ["Reported irritation"],
    "yotpo_777854447": ["Reported breakouts", "Reported irritation"],
    "yotpo_714243381": ["Reported breakouts"],
    "yotpo_711650901": ["Product identity confusion"],
    "yotpo_701490696": ["Reported irritation"],
    "yotpo_698667990": [
        "Reported irritation",
        "Texture dissatisfaction",
        "Refund/service difficulty",
        "Perceived lack of benefit"
    ]
}

assert set(review_codes) == set(cosrx_serum_low_reviews["review_id"])

cosrx_serum_review_themes = pd.DataFrame([
    {
        "review_id": review_id,
        "product_id": "COSRX_SERUM_01",
        "theme": theme,
        "coding_method": "Manual review of full text"
    }
    for review_id, themes in review_codes.items()
    for theme in themes
])

print("THEMES IN THESE EIGHT SELECTED REVIEWS")
print(
    cosrx_serum_review_themes.groupby("theme")["review_id"]
    .nunique()
    .sort_values(ascending=False)
    .to_string()
)

print("\nVERIFICATION STATUS BY INCENTIVE GROUP — ALL 300 SERUM REVIEWS")

serum_audit = analysis_reviews[
    analysis_reviews["product_id"] == "COSRX_SERUM_01"
].copy()

serum_audit["verification_group"] = (
    serum_audit["verified_standardized"]
    .astype("string")
    .fillna("Unknown")
)

print(
    serum_audit.groupby(["incentive_group", "verification_group"])
    .agg(
        reviews=("review_id", "size"),
        average_rating=("rating", "mean"),
        low_rating_count=("rating", lambda x: x.le(2).sum())
    )
    .round(2)
    .to_string()
)

In [ ]:
# Compare the time windows represented by each product's 300 reviews.
# Use calendar dates only; do not assign a timezone to Yotpo timestamps.
window_rows = []

for product_id, group in analysis_reviews.groupby("product_id"):
    parsed_dates = [
        pd.to_datetime(value, errors="coerce")
        for value in group["review_date"]
    ]

    if any(pd.isna(value) for value in parsed_dates):
        raise ValueError(f"Invalid review date: {product_id}")

    calendar_dates = [value.date() for value in parsed_dates]
    oldest = min(calendar_dates)
    newest = max(calendar_dates)

    window_rows.append({
        "product_id": product_id,
        "brand": group["brand"].iloc[0],
        "product_name": group["product_name"].iloc[0],
        "review_count": len(group),
        "oldest_review": oldest,
        "newest_review": newest,
        "span_days": (newest - oldest).days
    })

review_windows = (
    pd.DataFrame(window_rows)
    .sort_values("span_days", ascending=False)
    .reset_index(drop=True)
)

print("TIME WINDOWS COVERED BY THE COLLECTED REVIEWS")
print(review_windows.to_string(index=False))

In [ ]:
# Use calendar dates without assigning a timezone.
comparison_start = pd.Timestamp("2026-08-02").date()
comparison_end = pd.Timestamp("2026-09-28").date()

dated_reviews = analysis_reviews.copy()

dated_reviews["review_calendar_date"] = dated_reviews["review_date"].map(
    lambda value: pd.Timestamp(value).date()
)

common_period_reviews = dated_reviews[
    dated_reviews["review_calendar_date"].between(
        comparison_start, comparison_end
    )
].copy()

common_period_summary = (
    common_period_reviews.groupby("product_id")
    .agg(
        reviews_in_period=("review_id", "size"),
        average_rating=("rating", "mean"),
        low_rating_count=("rating", lambda x: x.le(2).sum()),
        incentivized_pct=(
            "incentivized_standardized",
            lambda x: x.mean() * 100
        )
    )
    .reset_index()
)

# Include every product, even if it has no reviews in this period.
common_period_summary = products[
    ["product_id", "brand", "product_name"]
].merge(
    common_period_summary,
    on="product_id",
    how="left",
    validate="one_to_one"
)

for column in ["reviews_in_period", "low_rating_count"]:
    common_period_summary[column] = (
        common_period_summary[column].fillna(0).astype(int)
    )

common_period_summary = common_period_summary.sort_values(
    "reviews_in_period"
)

print(f"SHARED PERIOD: {comparison_start} to {comparison_end}")
print(common_period_summary.round(2).to_string(index=False))
print("\nTotal reviews in shared period:", len(common_period_reviews))

In [ ]:
# Count reported skin types by brand.
skin_type_counts = pd.crosstab(
    analysis_reviews["brand"],
    analysis_reviews["skin_type_standardized"]
)

reported_counts = skin_type_counts.sum(axis=1)
total_counts = analysis_reviews.groupby("brand").size()

skin_type_coverage = pd.DataFrame({
    "total_reviews": total_counts,
    "skin_type_reported": reported_counts
})

skin_type_coverage["missing"] = (
    skin_type_coverage["total_reviews"]
    - skin_type_coverage["skin_type_reported"]
)

skin_type_coverage["coverage_pct"] = (
    100 * skin_type_coverage["skin_type_reported"]
    / skin_type_coverage["total_reviews"]
)

# Percentages are among reviewers who reported a skin type.
skin_type_percentages = (
    skin_type_counts
    .div(reported_counts.replace(0, float("nan")), axis=0)
    .mul(100)
)

print("SKIN-TYPE COVERAGE")
print(skin_type_coverage.round(1).to_string())

print("\nSKIN-TYPE COUNTS")
print(skin_type_counts.to_string())

print("\nSKIN-TYPE SHARE AMONG REPORTED RESPONSES (%)")
print(skin_type_percentages.round(1).to_string())

In [ ]:
# Coverage and dry-skin representation by product.
skin_type_audit = analysis_reviews.copy()

skin_type_audit["skin_type_reported"] = (
    skin_type_audit["skin_type_standardized"].notna()
)

skin_type_audit["reports_dry_skin"] = (
    skin_type_audit["skin_type_standardized"]
    .eq("Dry")
    .fillna(False)
)

product_skin_summary = (
    skin_type_audit.groupby(
        ["product_id", "brand", "product_name"]
    )
    .agg(
        total_reviews=("review_id", "size"),
        skin_type_reported=("skin_type_reported", "sum"),
        dry_skin_reviews=("reports_dry_skin", "sum")
    )
    .reset_index()
)

product_skin_summary["coverage_pct"] = (
    100 * product_skin_summary["skin_type_reported"]
    / product_skin_summary["total_reviews"]
)

product_skin_summary["dry_share_of_reported_pct"] = (
    100 * product_skin_summary["dry_skin_reviews"]
    / product_skin_summary["skin_type_reported"].replace(
        0, float("nan")
    )
)

print(
    product_skin_summary.sort_values(
        ["brand", "dry_share_of_reported_pct"],
        ascending=[True, False]
    ).round(1).to_string(index=False)
)

In [ ]:
# Select Cream Skin reviews with self-reported dry skin.
cream_skin_dry_reviews = analysis_reviews[
    analysis_reviews["product_id"].eq("LANEIGE_TONER_01")
    & analysis_reviews["skin_type_standardized"].eq("Dry")
].copy()

print("Cream Skin reviews reporting dry skin:",
      len(cream_skin_dry_reviews))

print("\nRATING DISTRIBUTION")
print(
    cream_skin_dry_reviews["rating"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nINCENTIVE STATUS")
print(
    cream_skin_dry_reviews["incentive_group"]
    .value_counts(dropna=False)
    .to_string()
)

# Inspect up to three lower-rated and three five-star English reviews.
english_reviews = cream_skin_dry_reviews[
    cream_skin_dry_reviews["language"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("en")
    .fillna(False)
]

lower_rated = english_reviews[
    english_reviews["rating"].le(3)
].sort_values("review_date", ascending=False).head(3)

five_star = english_reviews[
    english_reviews["rating"].eq(5)
].sort_values("review_date", ascending=False).head(3)

for label, subset in [
    ("LOWER-RATED EXAMPLES", lower_rated),
    ("FIVE-STAR EXAMPLES", five_star)
]:
    print(f"\n{label}")
    if subset.empty:
        print("No matching reviews.")
    for _, review in subset.iterrows():
        print("\n" + "-" * 60)
        print("Review ID:", review["review_id"])
        print("Date:", review["review_date"])
        print("Rating:", review["rating"])
        print("Incentive group:", review["incentive_group"])
        print("Title:", review["review_title"])
        print("Text:", review["review_text"])

In [ ]:
# Record the observed disclosure without overwriting the platform flag.
review_disclosure_notes = pd.DataFrame([{
    "review_id": "yotpo_892276945",
    "disclosure_type": "Gifted product",
    "disclosure_text": "gifted by laneige",
    "platform_flag_conflict": True,
    "coding_method": "Manual reading of review text"
}])

print("VARIANTS FOR THE SIX EXAMPLES")
example_ids = [
    "yotpo_826684126",
    "yotpo_821115210",
    "yotpo_809844695",
    "yotpo_894985986",
    "yotpo_892337897",
    "yotpo_892276945"
]

print(
    cream_skin_dry_reviews.loc[
        cream_skin_dry_reviews["review_id"].isin(example_ids),
        ["review_id", "rating", "variant_size", "incentive_group"]
    ].to_string(index=False)
)

print("\nREMAINING REVIEW RATED 3 OR BELOW")

remaining = cream_skin_dry_reviews[
    cream_skin_dry_reviews["rating"].le(3)
    & ~cream_skin_dry_reviews["review_id"].isin(example_ids)
]

for _, review in remaining.iterrows():
    print("\nReview ID:", review["review_id"])
    print("Rating:", review["rating"])
    print("Variant:", review.get("variant_size"))
    print("Language:", review.get("language"))
    print("Incentive group:", review["incentive_group"])
    print("Title:", review["review_title"])
    print("Text:", review["review_text"])

In [ ]:
from pathlib import Path

cream_skin_codes = {
    "yotpo_826684126": [
        "Size dissatisfaction",
        "Insufficient product information",
        "Ambiguous product reference"
    ],
    "yotpo_821115210": [
        "Positive moisturizing experience",
        "Positive lightweight feel",
        "Dispensing waste",
        "Price/value dissatisfaction"
    ],
    "yotpo_809844695": [
        "Positive perceived product benefit",
        "Dispenser failure",
        "Product spillage"
    ],
    "yotpo_809775832": [
        "Reduced perceived benefit versus past use",
        "Preference for alternatives"
    ],
    "yotpo_894985986": [
        "Positive hydration experience",
        "Positive lightweight feel"
    ],
    "yotpo_892337897": [
        "Positive moisturizing experience"
    ],
    "yotpo_892276945": [
        "Mist convenience",
        "Positive moisturizing experience",
        "Gifted-product disclosure"
    ]
}

cream_skin_review_themes = pd.DataFrame([
    {
        "review_id": review_id,
        "product_id": "LANEIGE_TONER_01",
        "theme": theme,
        "coding_method": "Manual review of full text"
    }
    for review_id, themes in cream_skin_codes.items()
    for theme in themes
])

# Combine the two exploratory case studies.
selected_review_themes = pd.concat(
    [cosrx_serum_review_themes, cream_skin_review_themes],
    ignore_index=True
)

assert selected_review_themes["review_id"].isin(
    analysis_reviews["review_id"]
).all()

assert not selected_review_themes.duplicated(
    ["review_id", "theme"]
).any()

analysis_folder = (
    PROJECT_ROOT / "data" / "analysis"
)
analysis_folder.mkdir(parents=True, exist_ok=True)

selected_review_themes.to_csv(
    analysis_folder / "selected_review_themes.csv",
    index=False
)

review_disclosure_notes.to_csv(
    analysis_folder / "review_disclosure_notes.csv",
    index=False
)

print("Reviews manually coded:",
      selected_review_themes["review_id"].nunique())
print("Theme assignments:", len(selected_review_themes))
print("Disclosure notes:", len(review_disclosure_notes))
print("Saved to:", analysis_folder.resolve())

In [ ]:
from pathlib import Path

analysis_folder = (
    PROJECT_ROOT / "data" / "analysis"
)
analysis_folder.mkdir(parents=True, exist_ok=True)

summary_tables = {
    "product_rating_summary": product_summary,
    "ratings_by_incentive_status": incentive_summary,
    "review_time_windows": review_windows,
    "shared_period_rating_summary": common_period_summary,
    "skin_type_by_product": product_skin_summary
}

for name, table in summary_tables.items():
    file_path = analysis_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)

    saved = pd.read_csv(file_path)
    assert len(saved) == len(table), f"Row count mismatch: {name}"

    print(f"{name}.csv: {len(saved)} rows saved")

print("\nAnalysis summaries saved.")
print("Folder:", analysis_folder.resolve())